In [ ]:
import pandas as pd
import numpy as np

In [ ]:
# Select your NDD and the date
ndd = 'DEM'
date = 'JUNE_25_2026'

In [ ]:
#! pwd

In [ ]:
# Select the NDD case files created in step 01
cases = pd.read_csv(f'/home/jupyter/workspace/WORKSPACE_BUCKET/data/NDD_rule_of_two/{ndd}_with_tenure_{date}.csv')
cases

In [ ]:
#Load controls created in step 02
controls = pd.read_csv('/home/jupyter/workspace/WORKSPACE_BUCKET/data/CONTROLS_with_tenure_JUNE_25_2026.csv')
controls = controls.drop(columns = 'DEM_DATE')
controls

In [ ]:
# Combine cases and controls
df = pd.concat([cases, controls])

#Check to make sure no duplicate IDs
print(df.ID.value_counts())

df = df.sort_values(by = f'{ndd}_DATE')
df = df.drop_duplicates(subset = 'ID', keep = 'first')

#Check to make sure no duplicate IDs
print(df.ID.value_counts())

df

In [ ]:
#Check number of cases and controls
df[f'{ndd}_DATE'].isna().value_counts()

In [ ]:
# adding ICD10 codes

# Add ICD10 Codes

In [ ]:
condition_list = ['A00', 'A01', 'A02', 'A04', 'A05', 'A15', 'A17', 'A18', 'A19', 'A20', 'A22', 'A23', 'A24', 'A26', 'A27', 'A28', 'A30', 'A31', 'A32', 'A35', 'A36', 'A37', 'A38', 'A39', 'A40', 'A41', 'A42', 'A44', 'A46', 'A48', 'A49', 'A50', 'A51', 'A52', 'A53', 'A65', 'A66', 'A67', 'A68', 'A69', 'B90', 'B95', 'B96', 'B97', 'G00', 'G01', 'J09', 'J10', 'J11', 'J12', 'J13', 'J14', 'J15', 'J16', 'J17', 'J18', 'K00', 'K01', 'K02', 'K03', 'K04', 'K05', 'K06', 'K08', 'K09', 'K35', 'K36', 'K37', 'K38', 'M00', 'M02', 'O85', 'P36']
condition = 'A00'
test = pd.read_csv(f'/home/jupyter/workspace/WORKSPACE_BUCKET/data/ICD10_Codes/{condition}_with_date_2015.csv')
test = test.rename(columns = {'person_id':'ID', 'start_date': condition})
test = test[['ID', condition]]
test

In [ ]:
for condition in condition_list:
    test = pd.read_csv(f'/home/jupyter/workspace/WORKSPACE_BUCKET/data/ICD10_Codes/{condition}_with_date_2015.csv')
    test = test.rename(columns = {'person_id':'ID', 'start_date': condition})
    test = test[['ID', condition]]
    df = df.merge(test, left_on = 'ID', right_on = 'ID', how = 'left')

In [ ]:
df[f'{ndd}_DATE'].isna().value_counts()

In [ ]:
#Encode NDD to 1 or 0
df[ndd] = np.where(df[ndd + '_DATE'].isna(), 0, 1)

#GENETIC_SEX to 1 or 2
df.loc[df.sex_at_birth == 'Female', 'SEX'] = '0'
df.loc[df.sex_at_birth == 'Male', 'SEX'] = '1'

In [ ]:
# Because we only have reliable virus data since 2015, the longer the study could be is 9 years
START_DATE = '2015-01-01'

for code in condition_list:

    # df['cutoff_date10'] = pd.to_datetime(df['tenure_date']) - pd.DateOffset(years=10)
    # df['cutoff_date5'] = pd.to_datetime(df['tenure_date']) - pd.DateOffset(years=5)
        
    #Select drug data at ANY time before tenure
    df[f'QC0_{code}'] = np.where((pd.to_datetime(df[f'{code}']) < pd.to_datetime(df['tenure_date'])), 1, 0)

        # # Select drug data at 0–5 years before tenure
        # df[f'QC0_5_{code}_{group}'] = np.where(
        # (pd.to_datetime(df[f'{code}_first_rx_{group}']) < pd.to_datetime(df['tenure_date'])) &
        # (pd.to_datetime(df[f'{code}_first_rx_{group}']) >= pd.to_datetime(df['cutoff_date5'])), 1, 0)
        
        # # Select drug data at 5–10 years before tenure
        # df[f'QC5_10_{code}_{group}'] = np.where(
        # (pd.to_datetime(df[f'{code}_first_rx_{group}']) < pd.to_datetime(df['cutoff_date5'])) &
        # (pd.to_datetime(df[f'{code}_first_rx_{group}']) >= pd.to_datetime(df['cutoff_date10'])), 1, 0)

        # #Select data only 10+ years before study end
        # df[f'QC10_{code}_{group}'] = np.where((df[f'{code}_first_rx_{group}'] < df['cutoff_date10']), 1, 0)

In [ ]:
df

# Add genetic status - APOE

In [ ]:
apoe = pd.read_csv('/home/jupyter/workspace/WORKSPACE_BUCKET/data/other/APOE_genotypes')
#eliminate unknown samples
apoe = apoe[apoe['APOE_GENOTYPE'] != 'unknown']
apoe

In [ ]:
apoe.APOE_GENOTYPE.value_counts()

In [ ]:
apoe["APOE"] = apoe["APOE_GENOTYPE"].map({
    "e3/e4": 1,
    "e4/e4": 2
}).fillna(0).astype(int)

In [ ]:
apoe = apoe[['IID', 'APOE']]
apoe = apoe.rename(columns = {'IID':'ID'})
apoe

In [ ]:
df = df.merge(apoe, left_on = 'ID', right_on = 'ID', how = 'left')
df

In [ ]:
df.APOE.value_counts(dropna=False)

In [ ]:
df = df[~df['APOE'].isna()]
df

In [ ]:
date = 'JUNE_25_2026'
df.to_csv(f'/home/jupyter/workspace/WORKSPACE_BUCKET/data/coxfiles/coxfiles_ICD10/{ndd}_{date}_ready_cox.csv', header = True, index = False)